# 03. 専門家デモを作る（Command Job）

**対応するテキスト**: [docs/04_専門家デモを作る.md](../docs/04_専門家デモを作る.md)

**前提**: [01_setup_azureml.ipynb](01_setup_azureml.ipynb) が完了していること。

> [!WARNING]
> **このノートブックは Azure 上で実行検証していません。**
> 本ハンズオンの構築時に検証したのは**ローカル実行だけ**です。
> Azure ジョブの所要時間・費用・出力例は記載していません。**あなたの環境で確認してください。**

> ⚠ **サブスクリプション ID を書き込んだノートブックをコミットしないでください。**
> **出力セルも消してからコミットしてください。**

## 1. 何を作るのか

[../src/collect_demos.py](../src/collect_demos.py) は、
**[../src/scripted_expert.py](../src/scripted_expert.py) の手続きでロボットを動かし、その実演を保存する**スクリプトです。

| ファイル | 中身 | 使うところ |
|---|---|---|
| `demos.npz` | 専門家の軌跡（観測と行動の並び） | **BC / GAIL の入力** |
| `scores.json` | 一様ランダムと専門家の成績 | **正規化リターンの基準** |

> **専門家の学習は行いません。** スクリプトなので、ジョブは短時間で終わります。

> ⚠ **`expert_policy.zip` のようなファイルはありません。**
> [06 章](../docs/06_DAggerを試す.md) の DAgger は専門家本体を必要としますが、
> **本ハンズオンの専門家はコード**なので、ジョブのスナップショット（`code="../src"`）に含まれて一緒に送られます。

In [ ]:
# ============================================================
#  ここを自分の環境に書き換えてください（01 と同じ値）
# ============================================================
SUBSCRIPTION_ID = "<SUBSCRIPTION_ID>"
RESOURCE_GROUP = "<RESOURCE_GROUP>"
WORKSPACE_NAME = "<AML_WORKSPACE_NAME>"

COMPUTE_NAME = "cpu-cluster"
ENV_REF = "il-pickplace-env@latest"
DATA_ASSET_NAME = "il-pickplace-demos"
EXPERIMENT = "il-demos"

TAGS = {
    "project": "il-workshop",
    "owner": "<your-alias>",
    "delete-after": "<YYYY-MM-DD>",
}

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import mlflow

ml_client = MLClient(
    credential=DefaultAzureCredential(exclude_interactive_browser_credential=False),
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)
ws = ml_client.workspaces.get(WORKSPACE_NAME)
print("接続しました:", ws.name)

#  ローカルから MLflow の記録を読むには、追跡 URI の明示設定が必要
#  出典: https://learn.microsoft.com/azure/machine-learning/how-to-use-mlflow-configure-tracking?view=azureml-api-2
tracking_uri = getattr(ws, "mlflow_tracking_uri", None)
if tracking_uri is None:
    tracking_uri = (
        f"azureml://{ws.location}.api.azureml.ms/mlflow/v1.0"
        f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}"
        f"/providers/Microsoft.MachineLearningServices/workspaces/{WORKSPACE_NAME}"
    )
mlflow.set_tracking_uri(tracking_uri)
print("MLflow 追跡先を設定しました。")

## 2. Command Job を定義して投入する

**ジョブの出力先は `${{outputs.demos}}` として受け取ります。**

| 引数 | 値 | 理由 |
|---|---|---|
| `--n-episodes` | `800` | **[05 章](../docs/05_BCを動かす.md) の比較実験でデモ 800 件の条件を使う**ため |
| `--seed` | `0` | 再現性のため固定（`src/il_common.py` の `set_seed()` が `torch` まで固定します） |

In [ ]:
from azure.ai.ml import command, Output
from azure.ai.ml.constants import AssetTypes

COLLECT_COMMAND = (
    "python collect_demos.py"
    " --n-episodes 800"
    " --seed 0"
    " --output-dir ${{outputs.demos}}"
)

collect_job = command(
    code="../src",                     # このフォルダー全体がスナップショットとして保存される
    command=COLLECT_COMMAND,
    outputs=dict(demos=Output(type=AssetTypes.URI_FOLDER)),
    environment=ENV_REF,
    compute=COMPUTE_NAME,
    experiment_name=EXPERIMENT,
    display_name="collect_demos_pick_and_place",
    tags=TAGS,
)

returned_job = ml_client.jobs.create_or_update(collect_job)
print("ジョブ名 :", returned_job.name)
print("studio  :", returned_job.studio_url)

In [ ]:
ml_client.jobs.stream(returned_job.name)

job = ml_client.jobs.get(returned_job.name)
print("ステータス:", job.status, "（Completed なら成功）")

### ⚠ ここで失敗したら

| 症状 | 対処 |
|---|---|
| `ValueError: 専門家がランダム行動を上回っていません` | [docs/A1](../docs/A1_トラブルシューティング.md) の 3-3 |
| `ResourceNotFound: il-pickplace-env` | [01_setup_azureml.ipynb](01_setup_azureml.ipynb) の 4. を実行していません |
| PyBullet の初期化で失敗する | [01_setup_azureml.ipynb](01_setup_azureml.ipynb) の疎通確認ジョブから見直してください |

## 3. `uri_folder` のデータ資産として登録する

**名前とバージョンを付けて登録**すれば、以降のノートブックは `il-pickplace-demos@latest` と書くだけで参照できます。

> 出典（Microsoft 公式）: [ジョブでのデータへのアクセス](https://learn.microsoft.com/azure/machine-learning/how-to-read-write-data-v2?view=azureml-api-2)

In [ ]:
from azure.ai.ml.entities import Data

demos_data = Data(
    name=DATA_ASSET_NAME,
    #  完了したジョブの出力を、そのままデータ資産の実体にする
    path=f"azureml://jobs/{returned_job.name}/outputs/demos",
    type=AssetTypes.URI_FOLDER,
    description="il/PandaPickAndPlace-v0 のスクリプト専門家デモ（demos.npz / scores.json）",
    tags=TAGS,
)
registered = ml_client.data.create_or_update(demos_data)

print("登録しました:", registered.name, "version =", registered.version)
print("以降は次のように参照します: azureml:" + DATA_ASSET_NAME + "@latest")

## 4. 基準値を確認する

`collect_demos.py` は、**正規化リターンの基準になる値**を MLflow に記録しています。
**この 2 つの値がこの後のすべての比較の土台**になります。

In [ ]:
run = mlflow.get_run(returned_job.name)

print("=== パラメーター ===")
for k, v in sorted(run.data.params.items()):
    print(f"  {k:24s}: {v}")

print("\n=== メトリック ===")
for k, v in sorted(run.data.metrics.items()):
    print(f"  {k:24s}: {v}")

metrics = run.data.metrics
if "expert_mean" in metrics and "random_mean" in metrics:
    gap = metrics["expert_mean"] - metrics["random_mean"]
    print(f"\n正規化リターンの分母 (expert_mean - random_mean) = {gap:.2f}")
    print("  ← この値が 0 以下だと比較が成り立ちません。プラスであることを確認してください。")

## 5. どうなれば成功か

**ローカル（Windows / 2026-08-18）で同じスクリプトを実行したときの実測値**です。

| 方策 | 成功率 | 平均リターン | 標準偏差 |
|---|---|---|---|
| 一様ランダム | **0.00** | **-50.00** | 0.00 |
| **スクリプト専門家** | **1.00** | **-11.25** | 4.04 |

| 項目 | 値 |
|---|---|
| 800 エピソードの収集時間（8 並列 / ローカル CPU） | **約 35 秒** |

> **Azure 上では同じ数値になりません。** 確認してほしいのは
> **「専門家の成功率がランダムより明確に高い」**ことです。

### ⚠ リターンの読み方

報酬は **成功していれば 0 / していなければ -1**（疎な報酬）。エピソードは必ず 50 ステップなので、

- **リターン -50** = 一度も成功しなかった
- **リターン -11** = **11 ステップ目で成功した**（そこから先は 0 点）

**0 に近いほど「速く成功した」**という意味になります。

## ✅ チェックリスト

- [ ] ジョブが `Completed` で終わった
- [ ] **データ資産 `il-pickplace-demos` を登録した**
- [ ] `expert_mean` が `random_mean` より**十分に大きい**ことを確認した
- [ ] `expert_success_rate` が 1.0 に近いことを確認した

---

**次へ**: [docs/05_BCを動かす.md](../docs/05_BCを動かす.md) → [04_bc_job.ipynb](04_bc_job.ipynb)